# 0장 R 시작하기 (R로 하는 연구 분석)

보건통계학 기초 21장의 신세포암 의료비 자료(rcc_cost.csv)로 R의 기본을 익힙니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요.


## a. 값과 함수

### 셀 1. 값을 만들고 함수에 넣기

In [ ]:
1 + 2 * 3
surv5 <- c(exp = 24.1, weibull = 19.5, lnorm = 32.6)   # 5년 생존율(%) 세 개
surv5 / 100
mean(surv5)
surv5["weibull"]

## b. 패키지

### 셀 2. 패키지 설치와 불러오기

In [ ]:
install.packages("survival")     # 처음 한 번만 (Colab은 세션마다)
library(survival)
packageVersion("survival")

## c. 자료 불러와 살펴보기

### 셀 3. 자료 불러오기

In [ ]:
options(HTTPUserAgent = "Mozilla/5.0")   # 사이트가 프로그램의 기본 요청을 막을 때를 대비
BASE <- "https://socialp-ajou.tecentriq12.workers.dev/data/"
cost <- read.csv(paste0(BASE, "rcc_cost.csv"))
dim(cost)
head(cost, 3)

### 셀 4. 열의 형식과 요약

In [ ]:
str(cost)
summary(cost$cost)
table(cost$arm)
sum(is.na(cost))

### 셀 5. 범주형 변수(factor)와 교차표

In [ ]:
cost$arm <- factor(cost$arm, levels = c("B", "A"))   # 표준요법 B를 기준으로
levels(cost$arm)
table(cost$arm, cost$stage4)

## d. 고르고 요약하기

### 셀 6. 행과 열 고르기, 군별 요약 (기본 R)

In [ ]:
a_only <- cost[cost$arm == "A", c("pid", "age", "cost")]
nrow(a_only)
cost$cost_m <- cost$cost / 100          # 만원 → 백만원
tapply(cost$cost, cost$arm, mean)
aggregate(cost ~ arm, data = cost, FUN = median)

### 셀 7. dplyr로 군별 요약

In [ ]:
install.packages("dplyr")
library(dplyr)
cost |>
  group_by(arm) |>
  summarise(n = n(),
            mean_cost   = mean(cost),
            median_cost = median(cost),
            no_inpatient = mean(cost_inp == 0))

## e. 결과를 읽고 남기기

### 셀 8. 모형을 맞추고 이름만 쓰기

In [ ]:
fit <- lm(cost ~ arm + age + male + cci + stage4, data = cost)
fit

### 셀 9. summary()로 자세히 보기

In [ ]:
summary(fit)

### 셀 10. 신뢰구간과 강건 표준오차

In [ ]:
round(confint(fit), 1)
library(sandwich)
library(lmtest)
rob <- vcovHC(fit, type = "HC3")         # 이분산에 강건한 공분산 (21장과 같은 HC3)
round(coeftest(fit, vcov = rob)["armA", ], 3)
round(coefci(fit, vcov = rob)["armA", ], 1)

### 셀 11. 이름을 틀리면

In [ ]:
mean(cost$Cost)

### 셀 13. 결과를 파일로 남기기

In [ ]:
res <- data.frame(term = names(coef(fit)), estimate = coef(fit), confint(fit))
write.csv(res, "rcc_cost_lm.csv", row.names = FALSE)
read.csv("rcc_cost_lm.csv")[2, ]

## f. 과제 정답

### 셀 14. 과제 1 정답: 병기별 평균 비용

In [ ]:
cost |>
  group_by(arm, stage4) |>
  summarise(n = n(), mean_cost = round(mean(cost), 1), .groups = "drop")

### 셀 15. 과제 2 정답: 고비용 환자의 비율

In [ ]:
cost$high <- cost$cost > 5000            # 5,000만원 넘게 쓴 환자
tapply(cost$high, cost$arm, mean)
table(cost$arm, cost$high)

### 셀 16. 과제 3 정답: 기준범주를 A로

In [ ]:
fit_b <- lm(cost ~ arm + age + male + cci + stage4,
            data = transform(cost, arm = relevel(arm, ref = "A")))
coef(fit_b)["armB"]